In [42]:
# --- Standard library ---
import os
import re
import json
import time
import base64
import io
from pathlib import Path
from typing import TypedDict

# --- Third-party libraries ---
from dotenv import load_dotenv    # loads .env into environment variables
import chromadb                    # ChromaDB client, same persistent store built during ingestion
from rank_bm25 import BM25Okapi    # BM25 term-based retrieval, rebuilt here from chunks.json
import voyageai                    # Voyage client, used to embed the live user query
import cohere                      # Cohere client, used for reranking the fused candidate pool
from PIL import Image              # used to resize retrieved images before sending them to Claude vision

from langchain_anthropic import ChatAnthropic   # Claude wrapper, used for generation (LangSmith auto-traces this)
from langgraph.graph import StateGraph, END      # orchestration: wires retrieve -> rerank -> generate as a graph
from langsmith import traceable                  # decorator that makes a plain Python function show up as its
                                                   # own step in the LangSmith trace tree, with timing captured

# --- Load environment variables ---
load_dotenv()

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
VOYAGE_API_KEY = os.environ["VOYAGE_API_KEY"]
COHERE_API_KEY = os.environ["COHERE_API_KEY"]
# LANGCHAIN_API_KEY / LANGCHAIN_TRACING_V2 / LANGCHAIN_PROJECT are read directly by the
# langchain/langsmith libraries from the environment, no need to pass them explicitly.

# --- Initialize clients ---
voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)
cohere_client = cohere.Client(api_key=COHERE_API_KEY)

# Claude Sonnet via LangChain (not the raw Anthropic SDK) specifically so LangSmith's
# auto-instrumentation captures this call automatically, tokens, cost, latency, no
# manual logging needed, same mechanism used throughout this project's tracing story.
llm = ChatAnthropic(model="claude-sonnet-4-6", api_key=ANTHROPIC_API_KEY, temperature=0)

# --- Load ChromaDB (persisted by data_ingestion.ipynb, same path, no rebuilding needed) ---
chroma_client = chromadb.PersistentClient(path="./data/chroma_db")
text_collection = chroma_client.get_collection(name="text_chunks")
image_collection = chroma_client.get_collection(name="image_chunks")

# --- Rebuild BM25 from chunks.json (BM25 itself isn't persisted anywhere, unlike Chroma,
# so we reload the chunk list and rebuild the term-frequency index fresh each run --
# this is fast, pure local computation, no API calls, done in Step 8 of ingestion too) ---
with open("data/chunks.json", "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

def simple_tokenize(text):
    """Same tokenizer used during ingestion -- must match exactly, since BM25
    scores depend on consistent tokenization between indexing and querying."""
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_corpus = [simple_tokenize(chunk["text"]) for chunk in all_chunks]
bm25_index = BM25Okapi(tokenized_corpus)

print(f"Setup complete. Loaded {len(all_chunks)} chunks, {text_collection.count()} text vectors, "
      f"{image_collection.count()} image vectors. BM25 rebuilt.")

Setup complete. Loaded 12684 chunks, 12684 text vectors, 105 image vectors. BM25 rebuilt.


In [43]:
from langchain_anthropic import ChatAnthropic   # Claude wrapper, used for generation (LangSmith auto-traces this)
from langgraph.graph import StateGraph, END      # orchestration: wires retrieve -> rerank -> generate as a graph
from langsmith import traceable                  # decorator that makes a plain Python function show up as its
                                                   # own step in the LangSmith trace tree, with timing captured

# --- Load environment variables ---
load_dotenv()

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
VOYAGE_API_KEY = os.environ["VOYAGE_API_KEY"]
COHERE_API_KEY = os.environ["COHERE_API_KEY"]

In [44]:
# --- Initialize clients ---
voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)
cohere_client = cohere.Client(api_key=COHERE_API_KEY)

In [45]:
llm = ChatAnthropic(model="claude-sonnet-4-6", api_key=ANTHROPIC_API_KEY, temperature=0)

In [46]:
# --- Load ChromaDB (persisted by data_ingestion.ipynb, same path, no rebuilding needed) ---
chroma_client = chromadb.PersistentClient(path="./data/chroma_db")
text_collection = chroma_client.get_collection(name="text_chunks")
image_collection = chroma_client.get_collection(name="image_chunks")

In [47]:
# --- Rebuild BM25 from chunks.json (BM25 itself isn't persisted anywhere, unlike Chroma,
# so we reload the chunk list and rebuild the term-frequency index fresh each run --
# this is fast, pure local computation, no API calls, done in ingestion too) ---
with open("data/chunks.json", "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

def simple_tokenize(text):
    """Same tokenizer used during ingestion -- must match exactly, since BM25
    scores depend on consistent tokenization between indexing and querying."""
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_corpus = [simple_tokenize(chunk["text"]) for chunk in all_chunks]
bm25_index = BM25Okapi(tokenized_corpus)

print(f"Setup complete. Loaded {len(all_chunks)} chunks, {text_collection.count()} text vectors, "
      f"{image_collection.count()} image vectors. BM25 rebuilt.")

Setup complete. Loaded 12684 chunks, 12684 text vectors, 105 image vectors. BM25 rebuilt.


In [48]:
RRF_K = 60          # standard RRF damping constant -- higher K flattens the influence of rank position
TOP_K_PER_SOURCE = 15  # how many candidates each individual retriever contributes before fusion


def bm25_search(query, top_k=TOP_K_PER_SOURCE):
    """Term-based retrieval: tokenize the query the same way chunks were tokenized,
    score every chunk, return the top_k highest-scoring chunk indices."""
    query_tokens = simple_tokenize(query)
    scores = bm25_index.get_scores(query_tokens)
    top_indices = scores.argsort()[-top_k:][::-1]
    return [all_chunks[i]["chunk_id"] for i in top_indices]


def semantic_search(query, top_k=TOP_K_PER_SOURCE):
    """Semantic retrieval: embed the query via Voyage (input_type='query', a different
    embedding treatment than 'document', since queries and documents play different
    roles in similarity search), then find the nearest chunks in Chroma."""
    query_embedding = voyage_client.multimodal_embed(
        inputs=[[query]], model="voyage-multimodal-3", input_type="query"
    ).embeddings[0]
    results = text_collection.query(query_embeddings=[query_embedding], n_results=top_k)
    return results["ids"][0], query_embedding  # return the embedding too, so image_search can reuse it


IMAGE_TOP_K = 3           # how many images to retrieve per query
IMAGE_DISTANCE_CUTOFF = 1.2  # Chroma cosine distance; higher = less similar, tune this by
                              # inspecting real distances returned during testing -- images
                              # with no genuine relevance shouldn't be shown just to fill 3 slots


def image_search(query_embedding, top_k=IMAGE_TOP_K):
    """Finds the most relevant CHARTS/TABLES/PHOTOS for this query, using the SAME
    query embedding already computed for text semantic search (voyage-multimodal-3
    puts text and images in one shared vector space, so one embedding call covers
    both -- no need to embed the query twice).

    This is the piece that was missing before: the image_chunks collection was
    being populated during ingestion but never queried at retrieval time, so the
    multimodal embeddings existed but were never actually used for anything.
    """
    if image_collection.count() == 0:
        return []

    results = image_collection.query(query_embeddings=[query_embedding], n_results=top_k)

    images = []
    for i in range(len(results["ids"][0])):
        distance = results["distances"][0][i]
        if distance > IMAGE_DISTANCE_CUTOFF:
            continue  # not actually relevant, skip rather than force-include it
        metadata = results["metadatas"][0][i]
        images.append({
            "image_id": results["ids"][0][i],
            "ticker": metadata["ticker"],
            "year": metadata["year"],
            "page_num": metadata["page_num"],
            "label": metadata["label"],
            "image_path": metadata["image_path"],
            "distance": distance,
        })
    return images


def reciprocal_rank_fusion(ranked_lists, k=RRF_K):
    """Combines multiple ranked lists of chunk_ids into one fused ranking.
    Each chunk's RRF score is the sum of 1/(k + rank) across every list it appears in --
    a chunk ranked highly by BOTH retrievers scores higher than one only one retriever liked,
    and this works without needing BM25 and cosine similarity scores to be on the same scale."""
    fused_scores = {}
    for ranked_list in ranked_lists:
        for rank, chunk_id in enumerate(ranked_list):
            fused_scores[chunk_id] = fused_scores.get(chunk_id, 0) + 1 / (k + rank)
    return sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)


@traceable(name="hybrid_retrieval")  # shows as its own timed step in the LangSmith trace
def hybrid_retrieve(query):
    """Runs BM25 and semantic search (always both, no conditional routing), fuses
    their results via RRF, and returns the fused candidate chunks with source tags
    (so the UI can show which retriever(s) surfaced each result).

    Also runs image_search using the SAME query embedding semantic_search already
    computed, and returns relevant images as a SEPARATE list, not merged into the
    fused text ranking. Images can't be scored by Cohere's text reranker or fused
    via RRF alongside term-frequency/cosine-similarity text rankings the same way,
    they're a different kind of result entirely, so they stay their own channel
    through the rest of the pipeline (see generate_answer for how they're used).
    """
    bm25_ids = bm25_search(query)
    semantic_ids, query_embedding = semantic_search(query)
    images = image_search(query_embedding)

    fused = reciprocal_rank_fusion([bm25_ids, semantic_ids])

    chunks_by_id = {c["chunk_id"]: c for c in all_chunks}
    candidates = []
    for chunk_id, rrf_score in fused:
        if chunk_id not in chunks_by_id:
            continue
        chunk = chunks_by_id[chunk_id]
        candidates.append({
            **chunk,
            "rrf_score": rrf_score,
            "from_bm25": chunk_id in bm25_ids,
            "from_semantic": chunk_id in semantic_ids,
        })
    return candidates, images


# Quick sanity check
_test_candidates, _test_images = hybrid_retrieve("What was Nvidia's data center revenue?")
print(f"Hybrid retrieval returned {len(_test_candidates)} fused text candidates and "
      f"{len(_test_images)} relevant images for a test query.")
for c in _test_candidates[:3]:
    tags = ("BM25" if c["from_bm25"] else "") + ("+Semantic" if c["from_semantic"] else "")
    print(f"  [{c['ticker']} {c['year']} / {tags}] {c['text'][:80]}...")
for img in _test_images:
    print(f"  IMAGE [{img['ticker']} {img['year']} p{img['page_num']} / {img['label']}] "
          f"distance={img['distance']:.3f}")

Hybrid retrieval returned 24 fused text candidates and 0 relevant images for a test query.
  [NVDA 2023 / BM25+Semantic] Data Center revenue for fiscal year 2023 was $15.01 billion, up 41% from fiscal ...
  [NVDA 2024 / BM25+Semantic] Data Center revenue for fiscal year 2024 was $47.5 billion, up 217% from fiscal ...
  [NVDA 2024 / BM25+Semantic] •
NVIDIA accelerated computing reached the tipping point, with Data Center reven...


In [49]:
RERANK_TOP_N = 5  # how many chunks survive reranking to actually go into the LLM's context


@traceable(name="rerank")
def rerank_candidates(query, candidates, top_n=RERANK_TOP_N):
    """Sends the fused candidate pool to Cohere Rerank, keeps only the top_n most
    relevant chunks. Cuts what the generation step has to read, both for token cost
    and to avoid diluting the LLM's context with marginally-relevant chunks."""
    if not candidates:
        return []

    documents = [c["text"] for c in candidates]
    response = cohere_client.rerank(
        model="rerank-v3.5",
        query=query,
        documents=documents,
        top_n=min(top_n, len(documents)),
    )

    reranked = []
    for result in response.results:
        chunk = candidates[result.index]
        reranked.append({**chunk, "rerank_score": result.relevance_score})
    return reranked


# Quick sanity check, reusing the candidates from Step 2's test query
_test_reranked = rerank_candidates("What was Nvidia's data center revenue?", _test_candidates)
print(f"Reranked down to {len(_test_reranked)} chunks.")
for c in _test_reranked:
    print(f"  score={c['rerank_score']:.3f}  [{c['ticker']} {c['year']}] {c['text'][:80]}...")

Reranked down to 5 chunks.
  score=0.876  [NVDA 2023] Data Center revenue for fiscal year 2023 was $15.01 billion, up 41% from fiscal ...
  score=0.836  [NVDA 2024] Data Center revenue for fiscal year 2024 was $47.5 billion, up 217% from fiscal ...
  score=0.825  [NVDA 2024] across industry verticals access NVIDIA AI infrastructure both through the cloud...
  score=0.806  [NVDA 2024] Fiscal 2024 was an extraordinary year.  Revenue increased 126% year on year to $...
  score=0.775  [NVDA 2024] algorithms move to video transformers, and more cars are equipped with cameras, ...


In [ ]:
SYSTEM_PROMPT = """You are a financial research assistant. Answer the user's question using ONLY the provided context chunks from company annual reports. Every claim in your answer must be grounded in the context. If the context does not contain enough information to answer, say so plainly rather than guessing. When you use a fact from a chunk, mention which company and fiscal year it came from."""


# --- Live RAGAS scoring setup (faithfulness + answer relevancy only) ---
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.dataset_schema import SingleTurnSample
from langchain_core.embeddings import Embeddings
import asyncio

# Use Haiku (not Sonnet) as the RAGAS judge -- faithfulness/relevancy scoring is a
# mechanical judging task, not nuanced generation, and RAGAS internally makes SEVERAL
# calls per metric (faithfulness decomposes the answer into individual claims and
# verifies each one separately; answer_relevancy generates multiple paraphrased
# questions to compare against), so this call-count multiplies fast. Haiku is
# roughly 5x cheaper ($1/$5 vs $3/$15 per million tokens) and keeps scoring cost
# from dominating the actual cost of the pipeline it's supposed to be evaluating.
ragas_judge_llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=ANTHROPIC_API_KEY, temperature=0)
ragas_llm = LangchainLLMWrapper(ragas_judge_llm)


class VoyageEmbeddingsForRagas(Embeddings):
    """Minimal LangChain-Embeddings-compatible adapter over Voyage, so RAGAS's
    answer_relevancy metric (which needs an embeddings model internally) uses
    the same embedding provider as the rest of this project instead of
    defaulting to OpenAI."""

    def embed_documents(self, texts):
        result = voyage_client.multimodal_embed(
            inputs=[[t] for t in texts], model="voyage-multimodal-3", input_type="document"
        )
        return result.embeddings

    def embed_query(self, text):
        result = voyage_client.multimodal_embed(
            inputs=[[text]], model="voyage-multimodal-3", input_type="query"
        )
        return result.embeddings[0]


ragas_embeddings = LangchainEmbeddingsWrapper(VoyageEmbeddingsForRagas())

# Bind the judge LLM/embeddings onto the metric objects once, up front,
# rather than passing them on every single scoring call.
faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_embeddings


def score_answer_live(query, answer, reranked_chunks):
    """Scores ONE live query's answer with faithfulness and answer relevancy.
    Unlike offline batch evaluation, this needs no ground truth, and genuinely
    changes based on whatever the user actually asked and what was retrieved
    for it, this is what makes these two metrics 'dynamic' rather than fixed
    numbers from a one-time test-set run."""
    sample = SingleTurnSample(
        user_input=query,
        response=answer,
        retrieved_contexts=[c["text"] for c in reranked_chunks],
    )

    async def _score():
        f_score = await faithfulness.single_turn_ascore(sample)
        r_score = await answer_relevancy.single_turn_ascore(sample)
        return f_score, r_score

    faithfulness_score, relevancy_score = asyncio.run(_score())
    return {"faithfulness": faithfulness_score, "answer_relevancy": relevancy_score}


def build_context_block(reranked_chunks):
    """Formats the reranked chunks into the context block the LLM sees.
    Only includes the metadata fields needed for citation (company, year, section) --
    internal fields like chunk_id, rrf_score, rerank_score stay out of the prompt,
    since they cost tokens without helping the LLM answer the question."""
    parts = []
    for c in reranked_chunks:
        parts.append(f"[{c['ticker']} FY{c['year']}, {c['section']}]\n{c['text']}")
    return "\n\n---\n\n".join(parts)


def prepare_image_for_generation(image_path, max_dimension=1568, jpeg_quality=85):
    """Resizes and re-encodes a retrieved image as base64 JPEG before sending it to
    Claude. Same resizing logic used during ingestion (Step 6/10) -- images can be
    well over Claude's 10 MB per-image API limit at full extracted resolution, and
    resolution beyond ~1568px on the long side adds cost without adding accuracy."""
    with open(image_path, "rb") as f:
        img = Image.open(f)
        img.load()
    if img.mode != "RGB":
        img = img.convert("RGB")
    if max(img.size) > max_dimension:
        scale = max_dimension / max(img.size)
        img = img.resize((int(img.width * scale), int(img.height * scale)), Image.LANCZOS)
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=jpeg_quality)
    return base64.standard_b64encode(buffer.getvalue()).decode("utf-8")


def build_image_content_blocks(images):
    """Turns retrieved image metadata into actual image content blocks Claude can
    see, this is the piece that makes the multimodal embeddings actually useful:
    without this, retrieved charts would just be metadata nobody looks at. Claude
    Sonnet is vision-capable, so it can genuinely read figures off a chart image,
    not just rely on a short caption describing it."""
    blocks = []
    for img in images:
        image_b64 = prepare_image_for_generation(img["image_path"])
        blocks.append({
            "type": "text",
            "text": f"[Chart/table from {img['ticker']} FY{img['year']}, page {img['page_num']}]",
        })
        blocks.append({
            "type": "image",
            "source": {"type": "base64", "media_type": "image/jpeg", "data": image_b64},
        })
    return blocks


@traceable(name="generate")
def generate_answer(query, reranked_chunks, images=None):
    """Calls Claude Sonnet with the reranked text context AND any retrieved images
    (actual chart/table content, not just captions). Uses Anthropic's prompt caching
    (cache_control on the system prompt) since SYSTEM_PROMPT is identical on every call --
    Anthropic can reuse its cached processing of that block instead of reprocessing it
    from scratch each time, cutting cost/latency on the (larger, static) system prompt."""
    images = images or []
    context_block = build_context_block(reranked_chunks)

    # Content passed as a list of blocks (rather than a plain string) is what lets us
    # attach cache_control to just the system prompt specifically.
    system_content = [{
        "type": "text",
        "text": SYSTEM_PROMPT,
        "cache_control": {"type": "ephemeral"},
    }]

    # User message is now a list of content blocks: the text context/question,
    # followed by any retrieved images (actual pixels, not descriptions of them),
    # so Claude can genuinely read figures off a chart rather than only relying
    # on whatever a caption or nearby paragraph happened to say about it.
    user_content = [{"type": "text", "text": f"Context:\n\n{context_block}\n\nQuestion: {query}"}]
    user_content.extend(build_image_content_blocks(images))

    response = llm.invoke(
        [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
        ]
    )

    usage = response.response_metadata.get("usage", {})
    answer_text = response.content

    # Score THIS specific answer live -- genuinely changes per query, since
    # it depends on what was actually asked and what was actually retrieved.
    live_scores = score_answer_live(query, answer_text, reranked_chunks)

    return {
        "answer": answer_text,
        "input_tokens": usage.get("input_tokens", 0),
        "output_tokens": usage.get("output_tokens", 0),
        "faithfulness": live_scores["faithfulness"],
        "answer_relevancy": live_scores["answer_relevancy"],
    }


# Quick sanity check -- now passing the images found earlier in Step 2
_test_result = generate_answer("What was Nvidia's data center revenue?", _test_reranked, _test_images)
print("Answer:", _test_result["answer"])  # full answer, no truncation
print(f"Images included in this call: {len(_test_images)}")
print(f"Tokens: {_test_result['input_tokens']} in / {_test_result['output_tokens']} out")
print(f"Faithfulness: {_test_result['faithfulness']:.2f} / 1.00")
print(f"Answer relevancy: {_test_result['answer_relevancy']:.2f} / 1.00")

C:\Users\aakas\AppData\Local\Temp\ipykernel_27924\3514754.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\aakas\AppData\Local\Temp\ipykernel_27924\3514754.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\aakas\AppData\Local\Temp\ipykernel_27924\3514754.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  rag

Answer: Based on the provided context, here is NVIDIA's Data Center revenue across the fiscal years mentioned:

- **FY2022**: Not explicitly stated, but implied as a baseline
- **FY2023**: **$15.01 billion**, up 41% from FY2022 *(NVDA FY2023, market_platform_highlights)*
- **FY2024**: **$47.5 billion**, up 217% from FY2023 *(NVDA FY2024, market_platform_highlights)*

Some additional context for FY2024 *(NVDA FY2024, revenue)*:
- **Data Center compute revenue** was up **244%** for the fiscal year
- **Networking revenue** was up **133%** for the fiscal year
- Approximately **40%** of Data Center revenue in FY2024 was attributed to **AI inference**
Images included in this call: 0
Tokens: 725 in / 207 out
Faithfulness: 1.00 / 1.00
Answer relevancy: 0.70 / 1.00


In [51]:
class PipelineState(TypedDict):
    """Shared state passed between graph nodes. Each node reads what it needs and
    adds its own output back into the state for the next node to use."""
    query: str
    candidates: list
    reranked: list
    images: list                # relevant charts/tables found by image_search
    answer: str
    input_tokens: int
    output_tokens: int
    faithfulness: float        # live per-query score, changes with every query
    answer_relevancy: float    # live per-query score, changes with every query


def retrieve_node(state: PipelineState) -> PipelineState:
    candidates, images = hybrid_retrieve(state["query"])
    return {**state, "candidates": candidates, "images": images}


def rerank_node(state: PipelineState) -> PipelineState:
    reranked = rerank_candidates(state["query"], state["candidates"])
    return {**state, "reranked": reranked}


def generate_node(state: PipelineState) -> PipelineState:
    # images pass straight through from retrieve_node to generation, they aren't
    # reranked (Cohere reranks text, not images), they're a separate channel
    result = generate_answer(state["query"], state["reranked"], state.get("images", []))
    return {
        **state,
        "answer": result["answer"],
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "faithfulness": result["faithfulness"],
        "answer_relevancy": result["answer_relevancy"],
    }


graph_builder = StateGraph(PipelineState)
graph_builder.add_node("retrieve", retrieve_node)
graph_builder.add_node("rerank", rerank_node)
graph_builder.add_node("generate", generate_node)

graph_builder.set_entry_point("retrieve")
graph_builder.add_edge("retrieve", "rerank")
graph_builder.add_edge("rerank", "generate")
graph_builder.add_edge("generate", END)

rag_graph = graph_builder.compile()

print("LangGraph pipeline compiled: retrieve -> rerank -> generate.")

LangGraph pipeline compiled: retrieve -> rerank -> generate.


In [52]:
def run_query(query, log_lines=None):
    """Runs one query through the full pipeline and prints a readable summary:
    the answer, its sources (with which retriever(s) surfaced each one), token
    counts, and end-to-end latency.

    If log_lines is provided (a list), the same output is also appended there,
    so it can be written to a file afterward. This avoids VS Code/Jupyter's
    cell output size limit truncating long results when running several
    queries in one cell -- the printed cell output can get cut off, but the
    file on disk always has everything in full.
    """
    start = time.time()
    result = rag_graph.invoke({"query": query})
    elapsed = time.time() - start

    lines = []
    lines.append(f"Q: {query}\n")
    lines.append(f"A: {result['answer']}\n")
    lines.append("Sources:")
    for c in result["reranked"]:
        tags = ("BM25" if c["from_bm25"] else "") + ("+Semantic" if c["from_semantic"] else "")
        lines.append(f"  [{c['ticker']} FY{c['year']} / {c['section']} / {tags}] "
                      f"rerank_score={c['rerank_score']:.3f}")

    images = result.get("images", [])
    if images:
        lines.append(f"\nImages used ({len(images)}):")
        for img in images:
            lines.append(f"  [{img['ticker']} FY{img['year']} p{img['page_num']} / {img['label']}] "
                          f"distance={img['distance']:.3f}")
    else:
        lines.append("\nImages used: none relevant found")

    lines.append(f"\nTokens: {result['input_tokens']} in / {result['output_tokens']} out")
    lines.append(f"Faithfulness: {result['faithfulness']:.2f} / 1.00")
    lines.append(f"Answer relevancy: {result['answer_relevancy']:.2f} / 1.00")
    lines.append(f"Latency: {elapsed:.2f}s")
    lines.append("=" * 70)

    for line in lines:
        print(line)
    if log_lines is not None:
        log_lines.extend(lines)

    # Append this query's live scores to a running log, so the Gradio Analytics
    # tab can show a rolling average across REAL queries actually asked, rather
    # than a fixed number from a one-time offline test-set run.
    log_entry = {
        "query": query,
        "faithfulness": result["faithfulness"],
        "answer_relevancy": result["answer_relevancy"],
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "latency_seconds": elapsed,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open("data/live_query_log.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(log_entry) + "\n")

    return result


# A handful of real test queries covering different query types:
# factual lookup, conceptual/paraphrased, and cross-document comparison.
TEST_QUERIES = [
    "What was Nvidia's data center revenue?",
    "How did Microsoft describe its approach to AI infrastructure investment?",
    "What risks did Target mention related to supply chain?",
    "How did Coca-Cola's leadership discuss inflation and economic conditions?",
]

# Collect full, untruncated output in a file in addition to printing it,
# so nothing is lost even if the notebook cell's displayed output gets cut off.
_log_lines = []
for q in TEST_QUERIES:
    run_query(q, log_lines=_log_lines)

with open("data/test_query_results.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(_log_lines))

print(f"\nFull untruncated results also saved to data/test_query_results.txt")

Q: What was Nvidia's data center revenue?

A: Based on the provided context, here is NVIDIA's Data Center revenue for the available fiscal years:

- **FY2022**: Not explicitly stated, but implied as a reference point
- **FY2023**: **$15.01 billion**, up 41% from fiscal year 2022 (per NVDA FY2023 market platform highlights)
- **FY2024**: **$47.5 billion**, up 217% from fiscal year 2023 (per NVDA FY2024 market platform highlights)

Some additional context for FY2024:
- **Data Center compute revenue** was up **244%** for the fiscal year
- **Networking revenue** was up **133%** for the fiscal year
- Approximately **40%** of Data Center revenue was attributed to **AI inference**
- In Q4 FY2024, **large cloud providers represented more than half** of Data Center revenue

Sources:
  [NVDA FY2023 / market_platform_highlights / BM25+Semantic] rerank_score=0.876
  [NVDA FY2024 / market_platform_highlights / BM25+Semantic] rerank_score=0.836
  [NVDA FY2024 / revenue / BM25] rerank_score=0.825
  [

AnthropicInvalidRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'You have reached your specified API usage limits. You will regain access on 2026-10-01 at 00:00 UTC.'}, 'request_id': 'req_011CewDwkKjqKpYjc2we7y7N'}

In [ ]:
import pandas as pd

# Read every live-scored query logged so far and compute rolling averages --
# this is genuinely dynamic: rerun this cell after asking more questions
# through run_query() or the Gradio app, and these numbers will change.
log_path = Path("data/live_query_log.jsonl")

if log_path.exists():
    log_df = pd.read_json(log_path, lines=True)
    print(f"Live queries logged so far: {len(log_df)}")
    print(f"Average faithfulness: {log_df['faithfulness'].mean():.2f} / 1.00")
    print(f"Average answer relevancy: {log_df['answer_relevancy'].mean():.2f} / 1.00")
    print(f"Average latency: {log_df['latency_seconds'].mean():.2f}s")
    print(f"Average input tokens: {log_df['input_tokens'].mean():.0f}")
    print(f"Average output tokens: {log_df['output_tokens'].mean():.0f}")
else:
    print("No queries logged yet -- run some queries through run_query() first.")

Live queries logged so far: 4
Average faithfulness: 0.85 / 1.00
Average answer relevancy: 0.78 / 1.00
Average latency: 35.84s
Average input tokens: 652
Average output tokens: 292
